# Logical Clustering of 2-D Points

This notebook demonstrates unsupervised clustering with `sltn`.

## Problem

We generate 200 unlabelled two-dimensional points and assign them to four clusters using only logical constraints. No ground-truth cluster labels are used during training.

The learnable predicate

$$
C(x, c)
$$

represents the truth value that point $x$ belongs to cluster $c$.

The knowledge base contains four clauses:

$$
\begin{aligned}
&\forall x \; \exists c \; C(x,c), \\
&\forall c \; \exists x \; C(x,c), \\
&\forall c,x,y :  Close(x,y) \mid  \; (C(x,c) \leftrightarrow C(y,c)), \\
&\forall c,x,y : Far(x,y) \mid \; \lnot (C(x,c) \land C(y,c)).
\end{aligned}
$$

The first two clauses prevent trivial solutions by requiring every point to belong to some cluster and every cluster to contain at least one point. The last two clauses encode geometric structure. Nearby points should receive the same cluster assignment, while distant points should not be assigned to the same cluster.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset: Four Gaussian clusters

The synthetic dataset contains four Gaussian point clouds in two dimensions. The ground-truth labels are used only for visualization. Training receives the point coordinates and the logical constraints, but not the cluster identities.

In [ ]:
nr_clusters = 4
nr_per_cluster = 50
close_threshold   = torch.tensor(0.2, device=device,requires_grad=True)
distant_threshold = torch.tensor(0.5, device=device,requires_grad=True)

margin = 0.2
means = [
    np.random.uniform([-1 + margin, -1 + margin], [0 - margin, 0 - margin], 2),
    np.random.uniform([0 + margin,  -1 + margin], [1 - margin, 0 - margin], 2),
    np.random.uniform([-1 + margin,  0 + margin], [0 - margin, 1 - margin], 2),
    np.random.uniform([0 + margin,   0 + margin], [1 - margin, 1 - margin], 2),
]
cov = np.array([[[.01, 0], [0, .01]]] * nr_clusters)

cluster_data = {
    i: np.random.multivariate_normal(mean=means[i], cov=cov[i], size=nr_per_cluster)
    for i in range(nr_clusters)
}
data = np.concatenate([cluster_data[i] for i in range(nr_clusters)]).astype(np.float32)

for i in range(nr_clusters):
    plt.scatter(cluster_data[i][:, 0], cluster_data[i][:, 1])
plt.title("Synthetic clusters")
plt.show()

## 1. Signature

The signature declares the symbolic vocabulary for clustering.

It contains two sorts:

- `Point`, representing a two-dimensional coordinate;
- `Cluster`, representing one of the four cluster identifiers.

It also declares three predicates:

- `C : Point × Cluster → Bool`, the learnable cluster-membership predicate;
- `Close : Point × Point → Bool`, used to select nearby pairs;
- `Far : Point × Point → Bool`, used to select distant pairs.

The variables `x`, `y`, and `c` are declared so that formulas can be grounded directly from the interpretation.

In [ ]:
import sltn
from sltn import Signature, Interpretation, KB

sig     = Signature("clustering")
sig.sort("Point")    # 2-D data point in [-1,1]²
sig.sort("Cluster")  # 4-D one-hot cluster label
sig.predicate("C",     ["Point", "Cluster"])  # C(x, c) = prob that x ∈ cluster c
sig.predicate("Close", ["Point", "Point"])    # dist(x,y) < θ_close
sig.predicate("Far",   ["Point", "Point"])    # dist(x,y) > θ_far

# Declare variables 
sig.variable("x", "Point")
sig.variable("y", "Point")
sig.variable("c", "Cluster")

print(sig)

## 2. Formulas

The four formulas are closed first-order sentences. The coverage clauses are:

$$
\forall x \; \exists c \; C(x,c),
$$

$$
\forall c \; \exists x \; C(x,c).
$$

The geometric clauses use guarded universal quantification:

$$
\forall c,x,y :  Close(x,y) \mid  \; (C(x,c) \leftrightarrow C(y,c)), \\
$$
$$
\forall c,x,y : Far(x,y) \mid \; \lnot (C(x,c) \land C(y,c)).
$$

The guards `Close(x, y)` and `Far(x, y)` restrict aggregation to point pairs satisfying the corresponding geometric condition.

In [ ]:
# All four axioms as closed formulas
phi1 = sig.parse("forall x: exists c: C(x, c)")
phi2 = sig.parse("forall c: exists x: C(x, c)")
phi3 = sig.parse("forall c, x, y | Close(x, y): (C(x, c) <-> C(y, c))")
phi4 = sig.parse("forall c, x, y | Far(x, y): not(C(x, c) & C(y, c))")

phi_kb = KB([phi1, phi2, phi3, phi4])

for name, phi in [("phi1", phi1), ("phi2", phi2), ("phi3", phi3), ("phi4", phi4)]:
    print(f"{name}: {phi}")

## 3. Neural model

The predicate `C` is grounded by a neural network. The model maps each two-dimensional point to a probability distribution over the four clusters using a softmax output layer. The predicate value `C(x, c)` is obtained by selecting the probability associated with cluster `c`. This makes cluster membership differentiable while still representing a categorical assignment.

In [ ]:
class ClusterMLP(torch.nn.Module):
    """Returns the probability that point x belongs to cluster c via softmax."""
    def __init__(self, sizes=(2, 16, 16, 16, 4)):
        super().__init__()
        self.elu     = torch.nn.ELU()
        self.softmax = torch.nn.Softmax(dim=1)
        self.layers  = torch.nn.ModuleList([
            torch.nn.Linear(sizes[i-1], sizes[i]) for i in range(1, len(sizes))
        ])

    def forward(self, x, c):
        for layer in self.layers[:-1]: x = self.elu(layer(x))
        probs = self.softmax(self.layers[-1](x))
        return torch.sum(probs * c, dim=-1, keepdim=True)

## 4. Interpretation

The interpretation grounds the symbolic vocabulary in tensors, neural modules, and fuzzy operators.

In this notebook:

- `Point` is interpreted as a two-dimensional tensor type;
- `Cluster` is interpreted as a four-dimensional one-hot tensor type;
- `C` is interpreted as the neural cluster classifier;
- `Close` and `Far` are interpreted as differentiable predicates based on Euclidean distance;
- the logical operators are configured with fuzzy semantics and aggregation operators.

The existential quantifier starts with a lenient aggregation setting and is later made stricter during training. This lets assignments form gradually before sharpening cluster membership.

In [ ]:
from sltn.interpretation.operators import aggregator, conjunction, implication, negation

interp = Interpretation(sig).to(device)

# Custom logic operators
interp.logic["∧"] = conjunction.AndProd()
interp.logic["¬"] = negation.NotStandard()
interp.logic["↔"] = implication.Equiv(conjunction.AndProd(), implication.ImpliesReichenbach())
interp.logic["∀"] = aggregator.AggregPMeanError(p=4)
interp.logic["∃"] = aggregator.AggregPMean(p=1, stable=True, eps=1e-6)
interp.logic["SatAgg"] = aggregator.AggregPMean(p=2)

# Distance-based predicates used as guards in the geometric clauses.
_dist = lambda xi, yi: torch.norm(xi - yi, dim=-1, keepdim=True)

interp["Point"] = sltn.Type("Point", shape=(2,))
interp["Cluster"] = sltn.Type("Cluster", shape=(4,))


interp["Close"] = lambda xi, yi: torch.nn.Sigmoid()(close_threshold -_dist(xi, yi))
interp["Far"]   = lambda xi, yi: torch.nn.Sigmoid()(_dist(xi, yi) - distant_threshold)
interp["C"] = ClusterMLP()

# Fixed data variables (no dataloader — full dataset each step); set .I once
interp["x"] = torch.tensor(data, device=device)
interp["y"] = torch.tensor(data, device=device)
interp["c"] = torch.tensor([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=torch.float32, device=device)

print(interp)

## 5. Training loop

Training optimizes the four-clause knowledge base. Each epoch evaluates the clauses under the current interpretation and updates:

- the parameters of the cluster-membership network;
- the close and far distance thresholds.

The existential aggregation parameter is scheduled during training:

- early epochs use a lenient setting to allow broad exploration;
- later epochs use a stricter setting to sharpen assignments.

The printed values report the satisfaction of each clause separately, making it possible to see which logical constraints are easier or harder to satisfy.

In [ ]:
from sltn import kb_describe, kb_backward

optimizer = torch.optim.Adam(list(interp["C"].parameters()) +
                             [close_threshold, distant_threshold], lr=0.01)
kb_dict = kb_describe(phi_kb)

for epoch in range(1000):
    p_exists = 1 if epoch <= 100 else 6
    interp.logic.Exists.p = p_exists

    optimizer.zero_grad()

    stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
    optimizer.step()

    if epoch % 200 == 0:
        sats_str = " | ".join(
            f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
        )
        w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
        print(f"epoch {epoch:4d} | [{sats_str}]")

## 6. Visualization

The visualization compares the generated Gaussian point clouds with the learned cluster-membership scores. The first panel shows the original synthetic clusters for reference. The remaining panels show the learned truth value of `C(x, c)` for each cluster `c`.

In [ ]:
x0, x1 = data[:, 0], data[:, 1]
cluster_labels = [[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]]

n_rows = (nr_clusters + 2) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(10, n_rows * 3))

axes[0, 0].set_title("Ground truth")
for i in range(nr_clusters):
    axes[0, 0].scatter(cluster_data[i][:, 0], cluster_data[i][:, 1])
axes[0, 1].axis('off')

data_t = torch.tensor(data, device=device)
for i, lbl in enumerate(cluster_labels):
    lbl_t = torch.tensor([lbl] * len(data), dtype=torch.float32, device=device)
    prC = interp["C"](data_t, lbl_t).detach().cpu().numpy()
    row, col = divmod(i, 2)
    axes[row + 1, col].set_title(f"C(x, cluster {i})")
    axes[row + 1, col].scatter(x0, x1, c=prC, vmin=0, vmax=1, cmap='viridis')

plt.tight_layout()
plt.show()

We print below the learned close and far thresholds used by the `Close` and `Far` predicates. These values determine which point pairs influence the guarded geometric clauses.

In [ ]:
print("Learned close threshold:", close_threshold.item())
print("Learned distant threshold:", distant_threshold.item())